In [ ]:
using KadanoffBaym
include("../src/wigner.jl") # Wigner transform

import SpecialFunctions: erf, erfc
import LinearAlgebra: diag

using Plots
default(fmt = :png)

## Generating a gaussian pulse

Consider a gaussian pulse
$$\xi(t) = \left(\frac{\Omega^2}{2 \pi}\right)^{1/4}\exp{\left(-\frac{\Omega^2}{4}(t-t_c)^2\right)}$$

Consider a modulator cavity given by the triple $(S_M, L_M, H_M)$

for which $L_M = \lambda(t) a$ where $\lambda(t) \triangleq \frac{\xi(t)}{\sqrt{\int_t^\infty \textrm{d}s |\xi(s)|^2}}$

In [ ]:
# Gaussian pulse parameters
const tc = 40.  # peak arrival time
const Ω = 0.12 # frequency bandwidth of the pulse

function ξ(t)
    return (Ω^2 / (2π))^(1/4) * exp(-(t - tc)^2 * (Ω^2 / 4))
end

# Pump and loss terms
const Γ = 2.
λ(t::Float64) = iszero(Γ) ? 0.0 : Γ * ξ(t) / sqrt(0.5 * erfc((t - tc) * Ω / sqrt(2)))
γ(t::Float64) = iszero(Γ) ? 0.0 : Γ * ξ(t)

# Photon field Hamiltonian
h(t::Float64) = 1e-2 - 0.5im * (λ(t) + γ(t))

# Artificial width of the peak
Σ = 1e-2

# Initial field occupation
n₀ = 0.0 

# Initial conditions
G₀ᵞL = GreenFunction(zeros(ComplexF64, 1, 1), Lesser)
G₀ᵞG = GreenFunction(zeros(ComplexF64, 1, 1), Greater)
G₀ᵞL[1,1] = -1.0im * n₀
G₀ᵞG[1,1] = -1.0im * (1 + n₀)

# Pack them
u₀ = [G₀ᵞL, G₀ᵞG];

# Define commutator
⋊(a,b) = a * b - b * adjoint(a)

function f_vert(u, times, t1, t2)
    t = times[t1]
    GL, GG = u[1], u[2]
    
    rhsL = -1.0im * ((h(t) + 1.0im * γ(t) - 1.0im * Σ) * GL[t1,t2])
    rhsG = -1.0im * ((h(t) + 1.0im * γ(t) - 1.0im * Σ) * GG[t1,t2])
    return [rhsL, rhsG]
end

function f_diag(u, times, t1, t2)
    t = times[t1]
    GL, GG = u[1], u[2]
    
    rhsL = -1.0im * (h(t) ⋊ GL[t1,t2] + 1.0im * γ(t) * (GL[t1,t2] + GG[t1,t2]))
    rhsG = -1.0im * (h(t) ⋊ GG[t1,t2] + 1.0im * λ(t) * (GG[t1,t2] - GL[t1,t2]) + 2.0im * γ(t) * GG[t1,t2])
    return [rhsL, rhsG]
end

In [ ]:
tspan = (0.0,100.0)

u, times = kbsolve(f_vert, f_diag, u₀, tspan[1], tspan[2], adaptive=true, atol=1e-9, rtol=1e-7)

# Extract photon Green functions
GᵞL = u[1];
GᵞG = u[2];

Plotting

In [ ]:
plot(times, λ.(times), label="Loss term")
plot!(times, γ.(times), label="Pump term")
plot!(twinx(), times, ξ.(times), color=:green, legend=:right, label="Pulse", line=:dot, xlabel=L"t")

In [ ]:
plot(times, GᵞL.data |> diag .|> imag .|> -, label="Lesser")
plot!(times, GᵞG.data |> diag .|> imag .|> -, label="Greater")
plot!(twinx(), times, ξ.(times), color=:green, legend=:right, label="Pulse", line=:dot, xlabel=L"t")

In [ ]:
heatmap(times, times, -imag(GᵞL.data))

In [ ]:
heatmap(times, times, -imag(GᵞG.data - GᵞL.data))

In [ ]:
# Time-depedent spectral function
ρ = wigner_transform(GᵞG.data - GᵞL.data);
wigner_plot!(real(ρ), times; ylims=(-3,3))